# 06. 모델 평가 (Evaluation)

## 학습 목표
- LLM 평가가 어려운 이유 이해
- 자동 지표 (Perplexity, BLEU, ROUGE) 직접 구현 및 이해
- 주요 벤치마크 (MMLU, HumanEval, MT-Bench, HELM) 이해
- LLM-as-Judge, 인간 평가 방법론 이해
- Fine-tuning 전후 비교 프레임워크 구축

## 참고 자료
- [MMLU (Hendrycks et al., 2021)](https://arxiv.org/abs/2009.03300)
- [Chatbot Arena (Zheng et al., 2023)](https://arxiv.org/abs/2306.05685)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate rouge-score nltk

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter
import math
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. LLM 평가가 어려운 이유

### 전통적 ML vs LLM 평가

| 전통적 ML | LLM |
|-----------|-----|
| 정답이 명확 (분류, 수치) | 좋은 답변이 여러 개 가능 |
| Accuracy, F1 등 객관적 지표 | 유용성, 안전성 등 주관적 기준 |
| 테스트셋으로 충분 | 새로운 태스크가 계속 등장 |
| 입출력 형식 고정 | 자유형 텍스트 출력 |

### LLM 평가의 핵심 어려움

1. **Open-ended Generation**: 같은 질문에 수많은 올바른 답이 존재
2. **Multi-dimensional Quality**: 정확성, 유용성, 안전성, 유창성 등 여러 차원
3. **Evaluation Contamination**: 학습 데이터에 벤치마크가 포함될 수 있음
4. **Format Sensitivity**: 프롬프트 형식에 따라 성능이 크게 변함
5. **Human Agreement**: 사람들끼리도 평가가 일치하지 않음

In [ ]:
# LLM 평가 방법론 분류 시각화
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 7))

# 카테고리 박스들
categories = [
    {'text': 'Automatic Metrics\n\nPerplexity\nBLEU / ROUGE\nExact Match', 
     'xy': (0.5, 3.5), 'color': '#E3F2FD', 'w': 3, 'h': 2.5},
    {'text': 'Benchmarks\n\nMMLU (knowledge)\nHumanEval (code)\nMT-Bench (chat)\nHELM (holistic)', 
     'xy': (4.5, 3.5), 'color': '#FFF3E0', 'w': 3, 'h': 2.5},
    {'text': 'LLM-as-Judge\n\nGPT-4 scoring\nPairwise comparison\nRubric-based', 
     'xy': (8.5, 3.5), 'color': '#F3E5F5', 'w': 3, 'h': 2.5},
    {'text': 'Human Evaluation\n\nElo Rating\nPairwise preference\nLikert scale', 
     'xy': (12.5, 3.5), 'color': '#E8F5E9', 'w': 3, 'h': 2.5},
]

for cat in categories:
    rect = mpatches.FancyBboxPatch(cat['xy'], cat['w'], cat['h'],
                                    boxstyle='round,pad=0.2',
                                    facecolor=cat['color'], edgecolor='gray', linewidth=2)
    ax.add_patch(rect)
    ax.text(cat['xy'][0] + cat['w']/2, cat['xy'][1] + cat['h']/2,
            cat['text'], ha='center', va='center', fontsize=9, fontweight='bold')

# 아래 축: 비용 vs 신뢰도 스펙트럼
ax.annotate('', xy=(15, 2.5), xytext=(0.5, 2.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(0.5, 2.0, 'Low Cost\nLow Reliability', fontsize=9, ha='left', color='gray')
ax.text(14, 2.0, 'High Cost\nHigh Reliability', fontsize=9, ha='right', color='gray')
ax.text(7.5, 2.0, 'Cost vs Reliability Tradeoff', fontsize=11, ha='center', fontweight='bold')

ax.set_xlim(-0.5, 16)
ax.set_ylim(1, 7)
ax.axis('off')
ax.set_title('LLM Evaluation Methods', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---
## 2. 자동 지표

### 2.1 Perplexity (당혹도)

모델이 텍스트를 얼마나 잘 예측하는지 측정. **낮을수록 좋다.**

$$\text{PPL}(W) = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i | w_{<i})\right)$$

직관: "다음 토큰을 예측할 때 평균적으로 몇 개의 선택지를 고려하는가"
- PPL = 1: 완벽하게 예측 (불가능에 가까움)
- PPL = 10: 평균 10개 선택지 중 하나
- PPL = 100: 매우 불확실

In [ ]:
def compute_perplexity(model, tokenizer, text, device='cpu'):
    """
    텍스트에 대한 Perplexity 계산.
    PPL = exp(-1/N * sum(log P(w_i | w_{<i})))
    """
    model.eval()
    inputs = tokenizer(text, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        # outputs.loss는 이미 평균 cross-entropy loss
        loss = outputs.loss
    
    perplexity = torch.exp(loss).item()
    return perplexity


# GPT-2로 Perplexity 계산
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

test_texts = [
    "The cat sat on the mat.",                          # 자연스러운 문장
    "Machine learning is a subset of artificial intelligence.",  # 일반적 지식
    "The banana flew across the purple mathematical sky.",      # 비정상적
    "asdf qwer zxcv tyui ghjk bnmp.",                          # 무의미한 문자
]

print("텍스트별 Perplexity (낮을수록 자연스러운 문장):")
print("=" * 65)
ppls = []
for text in test_texts:
    ppl = compute_perplexity(model, tokenizer, text, device)
    ppls.append(ppl)
    print(f"  PPL: {ppl:>10.2f} | {text}")

# 시각화
plt.figure(figsize=(10, 4))
colors = ['green' if p < 100 else 'orange' if p < 500 else 'red' for p in ppls]
plt.barh(range(len(test_texts)), ppls, color=colors)
plt.yticks(range(len(test_texts)), [t[:40] + '...' if len(t) > 40 else t for t in test_texts], fontsize=9)
plt.xlabel('Perplexity (lower is better)')
plt.title('Perplexity by Text')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### 2.2 BLEU Score

주로 기계 번역에서 사용. **생성된 텍스트가 참조 텍스트와 얼마나 n-gram이 겹치는지** 측정.

$$\text{BLEU} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

- $p_n$: n-gram precision (생성 텍스트의 n-gram 중 참조에도 있는 비율)
- $\text{BP}$: Brevity Penalty (너무 짧은 응답에 페널티)
- $w_n = 1/N$: 가중치 (보통 N=4, 균등 가중)

In [ ]:
def compute_ngrams(tokens, n):
    """n-gram 추출"""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def compute_bleu(reference, hypothesis, max_n=4):
    """
    BLEU Score 직접 구현.
    
    Args:
        reference: 참조 문장 (str)
        hypothesis: 생성된 문장 (str)
        max_n: 최대 n-gram (보통 4)
    """
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    # Brevity Penalty
    if len(hyp_tokens) < len(ref_tokens):
        bp = math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    else:
        bp = 1.0
    
    # n-gram precision
    log_precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(compute_ngrams(ref_tokens, n))
        hyp_ngrams = Counter(compute_ngrams(hyp_tokens, n))
        
        # Clipped count: 참조에 있는 만큼만 카운트
        clipped_count = 0
        total_count = 0
        for ngram, count in hyp_ngrams.items():
            clipped_count += min(count, ref_ngrams.get(ngram, 0))
            total_count += count
        
        if total_count == 0:
            precision = 0
        else:
            precision = clipped_count / total_count
        
        if precision > 0:
            log_precisions.append(math.log(precision))
        else:
            log_precisions.append(float('-inf'))
    
    # Geometric mean of precisions
    avg_log_precision = sum(log_precisions) / len(log_precisions)
    if avg_log_precision == float('-inf'):
        bleu = 0.0
    else:
        bleu = bp * math.exp(avg_log_precision)
    
    return bleu


# BLEU Score 테스트
reference = "The cat is sitting on the mat"
hypotheses = [
    "The cat is sitting on the mat",        # 완벽 일치
    "The cat sat on the mat",                # 비슷
    "A cat is on the mat",                   # 부분 일치
    "The dog is running in the park",        # 다른 내용
    "banana",                                 # 완전히 다름
]

print(f"Reference: '{reference}'")
print("=" * 55)
for hyp in hypotheses:
    bleu = compute_bleu(reference, hyp)
    print(f"  BLEU: {bleu:.4f} | '{hyp}'")

### 2.3 ROUGE Score

주로 요약 평가에 사용. **참조 텍스트의 n-gram 중 생성 텍스트에 포함된 비율** (recall 기반).

| ROUGE 종류 | 설명 |
|-----------|------|
| ROUGE-1 | Unigram 겹침 |
| ROUGE-2 | Bigram 겹침 |
| ROUGE-L | Longest Common Subsequence |

$$\text{ROUGE-N} = \frac{\text{reference의 n-gram 중 hypothesis에도 있는 수}}{\text{reference의 전체 n-gram 수}}$$

In [ ]:
def compute_rouge_n(reference, hypothesis, n=1):
    """
    ROUGE-N Score 직접 구현 (F1 score).
    """
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    ref_ngrams = Counter(compute_ngrams(ref_tokens, n))
    hyp_ngrams = Counter(compute_ngrams(hyp_tokens, n))
    
    # Overlap
    overlap = 0
    for ngram, count in ref_ngrams.items():
        overlap += min(count, hyp_ngrams.get(ngram, 0))
    
    # Precision, Recall, F1
    precision = overlap / max(sum(hyp_ngrams.values()), 1)
    recall = overlap / max(sum(ref_ngrams.values()), 1)
    
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    
    return {'precision': precision, 'recall': recall, 'f1': f1}


def compute_rouge_l(reference, hypothesis):
    """
    ROUGE-L Score (Longest Common Subsequence 기반).
    """
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    m, n = len(ref_tokens), len(hyp_tokens)
    
    # LCS 테이블
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_length = dp[m][n]
    precision = lcs_length / max(n, 1)
    recall = lcs_length / max(m, 1)
    
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    
    return {'precision': precision, 'recall': recall, 'f1': f1}


# ROUGE 테스트
reference = "Machine learning enables computers to learn from data and make predictions"
hypotheses = [
    "Machine learning enables computers to learn from data and make predictions",  # 완벽
    "Machine learning allows computers to learn from data",                          # 부분
    "Deep learning is a subset of machine learning",                                 # 약간 관련
    "The weather is nice today",                                                     # 무관
]

print(f"Reference: '{reference}'")
print("=" * 70)

for hyp in hypotheses:
    r1 = compute_rouge_n(reference, hyp, n=1)
    r2 = compute_rouge_n(reference, hyp, n=2)
    rl = compute_rouge_l(reference, hyp)
    print(f"\n  Hypothesis: '{hyp}'")
    print(f"    ROUGE-1 F1: {r1['f1']:.4f}")
    print(f"    ROUGE-2 F1: {r2['f1']:.4f}")
    print(f"    ROUGE-L F1: {rl['f1']:.4f}")

---
## 3. 벤치마크: MMLU, HumanEval, MT-Bench, HELM

### 주요 LLM 벤치마크 비교

| 벤치마크 | 평가 대상 | 형식 | 예시 |
|---------|---------|------|------|
| **MMLU** | 세계 지식 (57개 과목) | 4지선다 | "세포의 에너지 공장은? A)리보솜 B)미토콘드리아..." |
| **HumanEval** | 코드 생성 능력 | 함수 완성 | "def fibonacci(n):" → 구현 |
| **MT-Bench** | 대화 능력 | 멀티턴 대화 | 2턴 대화 후 GPT-4가 1-10점 채점 |
| **HELM** | 종합 평가 | 다양한 시나리오 | 정확성, 공정성, 독성 등 다차원 |
| **GSM8K** | 수학 추론 | 단계별 풀이 | 초등 수학 문제 |
| **TruthfulQA** | 사실성 | 자유 응답 | 미신, 오해에 대한 질문 |

In [ ]:
# MMLU 스타일 평가 시뮬레이션
# 실제 MMLU는 57개 과목, 15,000+ 문제

mmlu_examples = [
    {
        "question": "Which of the following is the powerhouse of the cell?",
        "choices": ["Ribosome", "Mitochondria", "Nucleus", "Golgi apparatus"],
        "answer": 1  # B: Mitochondria
    },
    {
        "question": "What is the capital of Australia?",
        "choices": ["Sydney", "Melbourne", "Canberra", "Brisbane"],
        "answer": 2  # C: Canberra
    },
    {
        "question": "Which sorting algorithm has the best average-case time complexity?",
        "choices": ["Bubble Sort O(n^2)", "Merge Sort O(n log n)", "Selection Sort O(n^2)", "Insertion Sort O(n^2)"],
        "answer": 1  # B: Merge Sort
    },
]


def evaluate_mmlu_style(model, tokenizer, question, choices, device='cpu'):
    """
    MMLU 스타일 평가: 각 선택지의 log probability를 비교.
    가장 높은 확률의 선택지를 모델의 답변으로 선택.
    """
    model.eval()
    choice_labels = ['A', 'B', 'C', 'D']
    
    prompt = f"Question: {question}\n"
    for i, choice in enumerate(choices):
        prompt += f"{choice_labels[i]}. {choice}\n"
    prompt += "Answer:"
    
    log_probs = []
    for label in choice_labels[:len(choices)]:
        full_text = prompt + f" {label}"
        inputs = tokenizer(full_text, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # 마지막 토큰(답변 레이블)의 log probability
        last_logits = logits[0, -1, :]
        answer_token_id = tokenizer.encode(f" {label}", add_special_tokens=False)[-1]
        log_prob = F.log_softmax(last_logits, dim=-1)[answer_token_id].item()
        log_probs.append(log_prob)
    
    predicted = np.argmax(log_probs)
    return predicted, log_probs


# MMLU 스타일 평가 실행
print("MMLU 스타일 평가 (GPT-2):")
print("=" * 60)

correct = 0
total = len(mmlu_examples)

for ex in mmlu_examples:
    predicted, log_probs = evaluate_mmlu_style(
        model, tokenizer, ex['question'], ex['choices'], device
    )
    is_correct = predicted == ex['answer']
    correct += is_correct
    
    choice_labels = ['A', 'B', 'C', 'D']
    print(f"\n  Q: {ex['question']}")
    for i, (choice, lp) in enumerate(zip(ex['choices'], log_probs)):
        marker = '>>>' if i == predicted else '   '
        correct_mark = ' (correct)' if i == ex['answer'] else ''
        print(f"  {marker} {choice_labels[i]}. {choice} (log_p={lp:.4f}){correct_mark}")
    print(f"  Predicted: {choice_labels[predicted]} | Correct: {choice_labels[ex['answer']]} | {'O' if is_correct else 'X'}")

print(f"\nAccuracy: {correct}/{total} ({correct/total*100:.1f}%)")

---
## 4. LLM-as-Judge: 다른 LLM으로 평가하는 방법

### 핵심 아이디어

강력한 LLM (예: GPT-4)을 사용하여 다른 모델의 출력을 평가하는 방법.

### 평가 방식

| 방식 | 설명 | 장점 |
|------|------|------|
| **Single Rating** | 1~10점 점수 매기기 | 간단, 절대 평가 |
| **Pairwise Comparison** | A vs B 중 더 좋은 것 선택 | 상대 비교에 효과적 |
| **Rubric-based** | 평가 기준표에 따라 채점 | 일관성 높음 |

### 예시 프롬프트 (Single Rating)

```
Please rate the following response on a scale of 1-10.

Question: {question}
Response: {response}

Criteria:
- Accuracy (1-10)
- Helpfulness (1-10)
- Clarity (1-10)

Please provide your rating and explanation.
```

### 주의점

- **Position Bias**: 첫 번째 응답을 선호하는 경향 → 순서를 바꿔서 2번 평가
- **Self-Enhancement Bias**: 자기 모델 출력을 높게 평가 → 다른 모델로 평가
- **Verbosity Bias**: 긴 응답을 더 좋게 평가하는 경향

In [ ]:
# LLM-as-Judge 프롬프트 생성 함수

def create_judge_prompt_single(question, response):
    """Single Rating 평가 프롬프트"""
    return f"""Please evaluate the following response to the given question.

Question: {question}

Response: {response}

Please rate the response on the following criteria (1-10):
1. Accuracy: Is the information correct?
2. Helpfulness: Does it answer the question well?
3. Clarity: Is the response clear and well-organized?

Format your response as:
Accuracy: [score]/10
Helpfulness: [score]/10
Clarity: [score]/10
Overall: [score]/10
Explanation: [brief explanation]"""


def create_judge_prompt_pairwise(question, response_a, response_b):
    """Pairwise Comparison 평가 프롬프트"""
    return f"""Compare the following two responses to the given question.

Question: {question}

Response A: {response_a}

Response B: {response_b}

Which response is better? Choose one:
- A is better
- B is better  
- Tie

Provide your choice and a brief explanation."""


# 프롬프트 예시
question = "What is transfer learning?"
response_good = "Transfer learning is a machine learning technique where a model trained on one task is adapted for a different but related task. This allows leveraging pre-learned features, reducing the need for large datasets and training time."
response_bad = "Transfer learning is when you transfer stuff."

print("=== Single Rating Prompt ===")
print(create_judge_prompt_single(question, response_good))
print("\n" + "=" * 60)
print("\n=== Pairwise Comparison Prompt ===")
print(create_judge_prompt_pairwise(question, response_good, response_bad))

print("\n" + "=" * 60)
print("\nNote: 실제 사용 시에는 이 프롬프트를 GPT-4 등 강력한 모델 API에 전달합니다.")
print("Colab에서 직접 실행하려면 OpenAI API key가 필요합니다.")

---
## 5. 인간 평가: Elo Rating, 방법론

### Chatbot Arena (LMSYS)

가장 유명한 인간 평가 시스템. 체스의 Elo Rating을 LLM에 적용.

1. 사용자가 질문을 입력
2. 익명의 두 모델이 각각 응답
3. 사용자가 더 좋은 응답 선택
4. Elo Rating 업데이트

### Elo Rating 수식

$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

$$R_A' = R_A + K(S_A - E_A)$$

- $E_A$: A가 이길 기대 확률
- $S_A$: 실제 결과 (1=승, 0=패, 0.5=무)
- $K$: 업데이트 크기 (보통 32)

In [ ]:
class EloRating:
    """Elo Rating 시스템 구현"""
    
    def __init__(self, k=32, initial_rating=1200):
        self.k = k
        self.initial_rating = initial_rating
        self.ratings = {}
        self.history = {}
    
    def get_rating(self, player):
        if player not in self.ratings:
            self.ratings[player] = self.initial_rating
            self.history[player] = [self.initial_rating]
        return self.ratings[player]
    
    def expected_score(self, rating_a, rating_b):
        """A가 B를 이길 기대 확률"""
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))
    
    def update(self, winner, loser, tie=False):
        """경기 결과로 레이팅 업데이트"""
        ra = self.get_rating(winner)
        rb = self.get_rating(loser)
        
        ea = self.expected_score(ra, rb)
        eb = self.expected_score(rb, ra)
        
        if tie:
            sa, sb = 0.5, 0.5
        else:
            sa, sb = 1, 0
        
        self.ratings[winner] = ra + self.k * (sa - ea)
        self.ratings[loser] = rb + self.k * (sb - eb)
        
        self.history[winner].append(self.ratings[winner])
        self.history[loser].append(self.ratings[loser])


# Elo Rating 시뮬레이션
np.random.seed(42)
elo = EloRating()

models = ['GPT-4', 'Claude-3', 'Gemini', 'LLaMA-3', 'Mistral']
# 모델 간 실력 (승률 결정용)
true_strength = {'GPT-4': 0.9, 'Claude-3': 0.85, 'Gemini': 0.8, 'LLaMA-3': 0.7, 'Mistral': 0.65}

# 시뮬레이션: 500번의 대결
for _ in range(500):
    # 랜덤으로 두 모델 선택
    m1, m2 = np.random.choice(models, 2, replace=False)
    
    # 실력 차이에 따라 승패 결정
    p_m1_wins = true_strength[m1] / (true_strength[m1] + true_strength[m2])
    
    if np.random.random() < p_m1_wins:
        elo.update(m1, m2)
    else:
        elo.update(m2, m1)

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Rating 변화 과정
ax = axes[0]
for model_name in models:
    ax.plot(elo.history[model_name], label=model_name, linewidth=1.5)
ax.set_xlabel('Match Number')
ax.set_ylabel('Elo Rating')
ax.set_title('Elo Rating Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 최종 Rating
ax = axes[1]
final_ratings = {m: elo.ratings[m] for m in models}
sorted_models = sorted(final_ratings, key=final_ratings.get, reverse=True)
sorted_ratings = [final_ratings[m] for m in sorted_models]
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.3, len(sorted_models)))

bars = ax.barh(range(len(sorted_models)), sorted_ratings, color=colors)
ax.set_yticks(range(len(sorted_models)))
ax.set_yticklabels(sorted_models)
ax.set_xlabel('Elo Rating')
ax.set_title('Final Elo Rankings')
ax.grid(True, alpha=0.3, axis='x')

for i, (bar, rating) in enumerate(zip(bars, sorted_ratings)):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{rating:.0f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. 평가 파이프라인 구축: Fine-tuning 전후 비교 프레임워크

실제 Fine-tuning 프로젝트에서 사용할 수 있는 비교 프레임워크를 구축한다.

```
동일 프롬프트 → Base Model 응답
              → Fine-tuned Model 응답
              → 자동 지표 비교 (PPL, BLEU, ROUGE)
              → Side-by-side 출력
```

In [ ]:
class EvaluationPipeline:
    """
    Fine-tuning 전후 비교 평가 파이프라인.
    """
    
    def __init__(self, base_model, finetuned_model, tokenizer, device='cpu'):
        self.base_model = base_model.to(device)
        self.ft_model = finetuned_model.to(device)
        self.tokenizer = tokenizer
        self.device = device
    
    def generate(self, model, prompt, max_new_tokens=100):
        """텍스트 생성"""
        model.eval()
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return full_text[len(prompt):].strip()
    
    def compute_perplexity(self, model, text):
        """Perplexity 계산"""
        model.eval()
        inputs = self.tokenizer(text, return_tensors='pt').to(self.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
        return torch.exp(outputs.loss).item()
    
    def evaluate_prompt(self, prompt, reference=None, max_new_tokens=100):
        """
        하나의 프롬프트에 대해 base vs fine-tuned 비교.
        """
        results = {}
        
        # 생성
        base_response = self.generate(self.base_model, prompt, max_new_tokens)
        ft_response = self.generate(self.ft_model, prompt, max_new_tokens)
        
        results['prompt'] = prompt
        results['base_response'] = base_response
        results['ft_response'] = ft_response
        
        # Perplexity (프롬프트 + 응답 전체)
        results['base_ppl'] = self.compute_perplexity(self.base_model, prompt + base_response)
        results['ft_ppl'] = self.compute_perplexity(self.ft_model, prompt + ft_response)
        
        # Reference가 있으면 ROUGE 계산
        if reference:
            results['base_rouge1'] = compute_rouge_n(reference, base_response, n=1)['f1']
            results['ft_rouge1'] = compute_rouge_n(reference, ft_response, n=1)['f1']
            results['base_rougeL'] = compute_rouge_l(reference, base_response)['f1']
            results['ft_rougeL'] = compute_rouge_l(reference, ft_response)['f1']
        
        return results
    
    def evaluate_batch(self, test_data, max_new_tokens=100):
        """
        여러 프롬프트에 대해 배치 평가.
        test_data: list of {'prompt': ..., 'reference': ...}
        """
        all_results = []
        for data in test_data:
            result = self.evaluate_prompt(
                data['prompt'],
                reference=data.get('reference'),
                max_new_tokens=max_new_tokens
            )
            all_results.append(result)
        return all_results
    
    def print_comparison(self, results):
        """결과를 보기 좋게 출력"""
        for r in results:
            print("=" * 70)
            print(f"Prompt: {r['prompt']}")
            print(f"\n  [Base Model]")
            print(f"    Response: {r['base_response'][:200]}")
            print(f"    PPL: {r['base_ppl']:.2f}")
            if 'base_rouge1' in r:
                print(f"    ROUGE-1: {r['base_rouge1']:.4f} | ROUGE-L: {r['base_rougeL']:.4f}")
            
            print(f"\n  [Fine-tuned Model]")
            print(f"    Response: {r['ft_response'][:200]}")
            print(f"    PPL: {r['ft_ppl']:.2f}")
            if 'ft_rouge1' in r:
                print(f"    ROUGE-1: {r['ft_rouge1']:.4f} | ROUGE-L: {r['ft_rougeL']:.4f}")
            print()

In [ ]:
# 평가 파이프라인 실행 예시
# (여기서는 동일한 GPT-2를 두 번 로드하여 데모; 실제로는 fine-tuned 모델 사용)

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained('gpt2')
ft_model = AutoModelForCausalLM.from_pretrained('gpt2')  # 실제로는 fine-tuned 모델

pipeline = EvaluationPipeline(base_model, ft_model, tokenizer, device)

test_data = [
    {
        'prompt': 'Machine learning is ',
        'reference': 'Machine learning is a branch of artificial intelligence that enables systems to learn from data.'
    },
    {
        'prompt': 'The transformer architecture ',
        'reference': 'The transformer architecture uses self-attention to process sequences in parallel.'
    },
]

results = pipeline.evaluate_batch(test_data, max_new_tokens=50)
pipeline.print_comparison(results)

In [ ]:
# 평가 결과 시각화 함수

def visualize_comparison(results):
    """Base vs Fine-tuned 비교 시각화"""
    
    metrics = {}
    
    # Perplexity
    base_ppls = [r['base_ppl'] for r in results]
    ft_ppls = [r['ft_ppl'] for r in results]
    
    has_rouge = 'base_rouge1' in results[0]
    
    if has_rouge:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    else:
        fig, axes = plt.subplots(1, 1, figsize=(6, 5))
        axes = [axes]
    
    # Perplexity 비교
    ax = axes[0]
    x = range(len(results))
    width = 0.35
    ax.bar([i - width/2 for i in x], base_ppls, width, label='Base', color='#FF9800', alpha=0.7)
    ax.bar([i + width/2 for i in x], ft_ppls, width, label='Fine-tuned', color='#4CAF50', alpha=0.7)
    ax.set_xlabel('Test Prompt')
    ax.set_ylabel('Perplexity (lower is better)')
    ax.set_title('Perplexity Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xticks(x)
    ax.set_xticklabels([f'P{i+1}' for i in x])
    
    if has_rouge:
        # ROUGE-1 비교
        ax = axes[1]
        base_r1 = [r['base_rouge1'] for r in results]
        ft_r1 = [r['ft_rouge1'] for r in results]
        ax.bar([i - width/2 for i in x], base_r1, width, label='Base', color='#FF9800', alpha=0.7)
        ax.bar([i + width/2 for i in x], ft_r1, width, label='Fine-tuned', color='#4CAF50', alpha=0.7)
        ax.set_xlabel('Test Prompt')
        ax.set_ylabel('ROUGE-1 F1 (higher is better)')
        ax.set_title('ROUGE-1 Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_xticks(x)
        ax.set_xticklabels([f'P{i+1}' for i in x])
        
        # ROUGE-L 비교
        ax = axes[2]
        base_rl = [r['base_rougeL'] for r in results]
        ft_rl = [r['ft_rougeL'] for r in results]
        ax.bar([i - width/2 for i in x], base_rl, width, label='Base', color='#FF9800', alpha=0.7)
        ax.bar([i + width/2 for i in x], ft_rl, width, label='Fine-tuned', color='#4CAF50', alpha=0.7)
        ax.set_xlabel('Test Prompt')
        ax.set_ylabel('ROUGE-L F1 (higher is better)')
        ax.set_title('ROUGE-L Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_xticks(x)
        ax.set_xticklabels([f'P{i+1}' for i in x])
    
    plt.tight_layout()
    plt.show()
    
    # 요약 통계
    print("\n평균 지표 비교:")
    print(f"  Perplexity - Base: {np.mean(base_ppls):.2f} | Fine-tuned: {np.mean(ft_ppls):.2f}")
    if has_rouge:
        print(f"  ROUGE-1    - Base: {np.mean(base_r1):.4f} | Fine-tuned: {np.mean(ft_r1):.4f}")
        print(f"  ROUGE-L    - Base: {np.mean(base_rl):.4f} | Fine-tuned: {np.mean(ft_rl):.4f}")


visualize_comparison(results)

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 종합 평가 파이프라인 구축

이전 노트북(02 또는 03)에서 Fine-tuning한 모델을 로드하여, base 모델과 비교 평가하세요.

1. Base GPT-2와 Fine-tuned GPT-2 로드
2. 5개 이상의 테스트 프롬프트 준비 (reference 포함)
3. EvaluationPipeline으로 평가 실행
4. 결과 시각화 + 분석
5. LLM-as-Judge 프롬프트도 생성하여 출력

In [ ]:
# TODO: 종합 평가 파이프라인
# Hint:
# 1. Fine-tuned 모델 로드
#    ft_model = AutoModelForCausalLM.from_pretrained('./gpt2-instruction-ft/checkpoint-...')
#    또는 LoRA 모델: PeftModel.from_pretrained(base_model, './gpt2-lora')
#
# 2. 테스트 데이터 준비
#    test_data = [
#        {'prompt': '...', 'reference': '...'},
#        ...
#    ]
#
# 3. 평가 실행
#    pipeline = EvaluationPipeline(base_model, ft_model, tokenizer, device)
#    results = pipeline.evaluate_batch(test_data)
#    pipeline.print_comparison(results)
#    visualize_comparison(results)
#
# 4. LLM-as-Judge 프롬프트 생성
#    for r in results:
#        judge_prompt = create_judge_prompt_pairwise(
#            r['prompt'], r['base_response'], r['ft_response']
#        )
#        print(judge_prompt)

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| LLM 평가의 어려움 | 정답이 여러 개, 주관적 기준 | 단일 지표로 충분하지 않음 |
| Perplexity | 다음 토큰 예측 불확실성 | 낮을수록 좋음, 언어 모델 기본 지표 |
| BLEU | n-gram precision (번역) | 높을수록 좋음, 참조 기반 |
| ROUGE | n-gram recall (요약) | 높을수록 좋음, 참조 기반 |
| MMLU | 57과목 4지선다 지식 평가 | 모델의 세계 지식 측정 |
| MT-Bench | GPT-4가 채점하는 대화 평가 | 2턴 대화 능력 |
| LLM-as-Judge | 강력한 LLM이 평가 | 비용 효율적이지만 편향 존재 |
| Elo Rating | 체스식 레이팅 (Chatbot Arena) | 인간 선호 기반 순위 |
| 평가 파이프라인 | 동일 프롬프트로 Base vs FT 비교 | 자동 지표 + 정성 평가 병행 |

---

## 06-finetuning-alignment 시리즈 완료!

이 시리즈에서 다룬 내용:

1. **Transfer Learning**: Feature Extraction vs Fine-tuning, Layer-wise LR
2. **Full Fine-tuning**: Instruction tuning, Alpaca format, Trainer API
3. **PEFT / LoRA**: Low-rank adaptation, QLoRA, 파라미터 효율성
4. **RLHF**: Reward Model, PPO, KL constraint
5. **DPO**: Reward-free alignment, 수학적 유도
6. **Evaluation**: Perplexity, BLEU, ROUGE, 벤치마크, LLM-as-Judge